<a href="https://colab.research.google.com/github/SemalDeSilva/TEAI/blob/Sooriyaarachchi-N.D-IT22254702-withering_time_predict/Tea_Withering_Time_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_PATH = "/content/drive/MyDrive/predict remainin time/withering time.csv"

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib


In [4]:
try:
    df = pd.read_csv(DATA_PATH)
except UnicodeDecodeError:
    df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset loaded successfully")
print("Rows:", len(df))
display(df.head())

Dataset loaded successfully
Rows: 262


,batch_id,rainy_day,timestamp,M0_g,Mt_g,MC0,temperature_C,humidity_percent,hydrometer_gap_C,elapsed_time_min,current_moisture_percent,remaining_time_to_well_withered_min,total_withering_time_min
0,N01,0,2025-11-07 22:00:00,102.26,102.16,0.741,28.98,62.10,5,0,74.07,600.0,600
1,N01,0,2025-11-07 22:30:00,102.26,98.91,0.741,29.10,62.00,5,30,73.22,570.0,600
2,N01,0,2025-11-07 23:00:00,102.26,95.32,0.741,29.20,61.60,5,60,72.21,540.0,600
3,N01,0,2025-11-07 23:30:00,102.26,92.29,0.741,28.74,62.62,5,90,71.30,510.0,600
4,N01,0,2025-11-08 00:00:00,102.26,89.29,0.741,28.91,61.65,5,120,70.34,480.0,600


In [5]:
# Calculate current_moisture_percent
# Md = M0_g * (1 - MC0)
# moisture% = ((Mt_g - Md) / Mt_g) * 100

def calculate_current_moisture(M0_g, Mt_g, MC0):
    Md = M0_g * (1 - MC0)
    return ((Mt_g - Md) / Mt_g) * 100

df["current_moisture_percent"] = calculate_current_moisture(
    df["M0_g"], df["Mt_g"], df["MC0"]
).round(2)

print("current_moisture_percent calculated using mass-balance equation")
display(df[["M0_g", "Mt_g", "MC0", "current_moisture_percent"]].head())

current_moisture_percent calculated using mass-balance equation


,M0_g,Mt_g,MC0,current_moisture_percent
0,102.26,102.16,0.741,74.07
1,102.26,98.91,0.741,73.22
2,102.26,95.32,0.741,72.21
3,102.26,92.29,0.741,71.30
4,102.26,89.29,0.741,70.34


In [6]:
# Keep Only 8h–12h Withering Data

LOWER_RATIO = 0.54
UPPER_RATIO = 0.56

df = df.sort_values(["batch_id", "elapsed_time_min"]).copy()
df["mass_ratio"] = df["Mt_g"] / df["M0_g"]

for batch_id, group in df.groupby("batch_id"):
    finished = False
    for idx in group.index:
        if LOWER_RATIO <= df.loc[idx, "mass_ratio"] <= UPPER_RATIO:
            finished = True
        if finished:
            df.loc[idx, "remaining_time_to_well_withered_min"] = 0

df = df.drop(columns=["mass_ratio"])

print("Mass based remaining time correction applied")


Mass based remaining time correction applied


In [7]:
# PART 7 — Select Features and Target

TARGET = "remaining_time_to_well_withered_min"

FEATURES = [
    "rainy_day",
    "M0_g",
    "Mt_g",
    "MC0",
    "temperature_C",
    "humidity_percent",
    "hydrometer_gap_C",
    "elapsed_time_min",
    "current_moisture_percent",
]

df_model = df[FEATURES + [TARGET]].dropna()

X = df_model[FEATURES]
y = df_model[TARGET]

print("Features & target prepared")
print("X shape:", X.shape)


Features & target prepared
X shape: (262, 9)


In [8]:
# PART 8 — Train / Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

print("Data split complete")

Data split complete


In [9]:
# PART 9 — Train Random Forest Model

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model trained successfully")

Model trained successfully


In [10]:
# PART 10 — Evaluate Model

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)

mse = mean_squared_error(y_test, pred)   # NO squared argument
rmse = np.sqrt(mse)                      # manual RMSE

r2 = r2_score(y_test, pred)

print("MAE  (minutes):", round(mae, 2))
print("RMSE (minutes):", round(rmse, 2))
print("R²:", round(r2, 4))


MAE  (minutes): 19.19
RMSE (minutes): 24.17
R²: 0.9846


In [11]:
# PART 11 — Save Model to Google Drive

MODEL_PATH = "/content/drive/MyDrive/predict remainin time/withering_time.pkl"
joblib.dump(model, MODEL_PATH)

print("\n Model saved to:", MODEL_PATH)


 Model saved to: /content/drive/MyDrive/predict remainin time/withering_time.pkl


In [16]:
# PART 12 — REAL-TIME PREDICTION (AUTO MOISTURE CALCULATION)
# Live input values
rainy_day = 0
M0_g = 95.19
Mt_g =61.85
MC0 = 0.718
temperature_C = 30.6
humidity_percent = 66.99
hydrometer_gap_C = 5
elapsed_time_min = 510

In [17]:
# Automatically calculate moisture
current_moisture_percent = calculate_current_moisture(M0_g, Mt_g, MC0)

In [18]:
# Prepare input row
new_sample = pd.DataFrame([{
    "rainy_day": rainy_day,
    "M0_g": M0_g,
    "Mt_g": Mt_g,
    "MC0": MC0,
    "temperature_C": temperature_C,
    "humidity_percent": humidity_percent,
    "hydrometer_gap_C": hydrometer_gap_C,
    "elapsed_time_min": elapsed_time_min,
    "current_moisture_percent": round(current_moisture_percent, 2)
}])

eta_min = float(model.predict(new_sample)[0])

In [19]:
if (Mt_g / M0_g) <= UPPER_RATIO:
    eta_min = 0.0

eta_min = max(0.0, eta_min)

print("\n REAL-TIME OUTPUT")
print("Calculated moisture (%):", round(current_moisture_percent, 2))
print("Remaining time (minutes):", round(eta_min, 1))
print("Remaining time (hours):", round(eta_min/60, 2))


 REAL-TIME OUTPUT
Calculated moisture (%): 56.6
Remaining time (minutes): 178.2
Remaining time (hours): 2.97
